In [ ]:
!nvidia-msi

In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics
from ultralytics import YOLO
import cv2
import numpy as np
import glob

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="PvEETTzxzsoQwlcklKW5")
project = rf.workspace("med-seg").project("planets-2_detection")
version = project.version(2)
dataset = version.download("yolov11")

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt")

# Train the model
train_results = model.train(
    data="/content/planets-2_detection-2/data.yaml",  
    epochs=100,  # number of training epochs
    imgsz=640,  # training image size
    device=0,  # device to run on, i.e. device=0 or device=0,1,2,3 or device=cpu
)

In [ ]:

import os
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
from IPython import display
from IPython.display import display, Image
Image(filename=f'/content/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
Image(filename=f'/content/runs/detect/train/results.png', width=600)

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

!yolo task=detect mode=val model=/content/runs/detect/train/weights/best.pt conf=0.25 source='/content/planets-2_detection-2/test/images' data='/content/planets-2_detection-2/data.yaml' hide_conf=True

In [ ]:
!yolo task=detect mode=predict model=/content/runs/detect/train/weights/best.pt conf=0.25 source='/content/planets-2_detection-2/test/images' hide_conf=True

In [ ]:
model = YOLO("/content/runs/detect/train/weights/best.pt")  # Replace with your model path

# Evaluate the model on the test dataset
results = model.val(
    data="/content/planets-2_detection-2/data.yaml",  # Path to your dataset YAML file
    split="test",         # Evaluate on the test set
    imgsz=640,            # Image size for inference
    conf=0.25,            # Confidence threshold
    iou=0.6,              # IoU threshold for NMS
    device="0",           # Use GPU (set to "cpu" for CPU)
    save_json=True,       # Save results to JSON for further analysis
    save_conf=True,       # Save confidence scores in the JSON file
)

# Access metrics
metrics = results.box  # Access the Metrics object

# Print metrics
print(f"mAP50-95: {metrics.map}")       # mAP@0.5:0.95
print(f"mAP50: {metrics.map50}")         # mAP@0.5
print(f"mAP75: {metrics.map75}")         # mAP@0.75
print(f"Precision: {metrics.p.mean()}")  # Mean precision across all classes
print(f"Recall: {metrics.r.mean()}")     # Mean recall across all classes
print(f"F1 Score: {metrics.f1.mean()}")  # Mean F1 score across all classes
#print(f"Precision: {metrics.Precision}") # Precision
#print(f"Recall: {metrics.recall}")       # Recall
#print(f"IoU: {metrics.iou}")       # IoU
#print(f"IoU: {metrics.miou}")       # MIoU
#print(f"IoU: {metrics.f1}")       # f1-score
#print(f"IoU: {metrics.accuracy}")       # accuracy



# Print precision, recall, and F1 score for each class
for i, class_name in enumerate(model.names):
    print(f"Class: {class_name}")
    print(f"  Precision: {metrics.p[i]}")
    print(f"  Recall: {metrics.r[i]}")
    print(f"  F1 Score: {metrics.f1[i]}")

In [ ]:
from ultralytics import YOLO
import glob
import os
from PIL import Image
from IPython.display import display

model = YOLO("/content/runs/detect/train/weights/best.pt")

test_folder = "/content/planets-2_detection-2/test/images"


image_paths = glob.glob(os.path.join(test_folder, "*.jpg"))  

# Run YOLO on all test images and hide confidence scores
results = model(image_paths, save=True, hide_conf=True)

for image_path in image_paths:
    # Path to saved result (YOLO saves results in 'runs/segment/predict/')
    result_image_path = os.path.join("runs/detect/predict2", os.path.basename(image_path))

    # Show the image
    display(Image.open(result_image_path))

In [ ]:
import os
import cv2
import numpy as np
import torch
from ultralytics import YOLO, SAM
from sklearn.metrics import accuracy_score

yolo_model = YOLO("/content/runs/detect/train/weights/best.pt")  
sam_model = SAM("sam2_b.pt")  

# Define paths for the folder of images and ground truth labels
images_folder = "/content/planets-2_detection-2/test/images"
labels_folder = "/content/planets-2_detection-2/test/labels"

all_yolo_metrics = {"precision": [], "recall": [], "f1": [], "accuracy": []}
all_sam_metrics = {"iou": [], "dice": [], "accuracy": [], "sensitivity": [], "specificity": [], "miou": [], "map50": []}

# Function to compute IoU between two bounding boxes
def compute_iou(box1, box2):
    """
    Compute Intersection over Union (IoU) between two bounding boxes.
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union != 0 else 0

# Function to convert polygon coordinates to a binary mask
def polygon_to_mask(polygon, width, height):
    """
    Convert a polygon (list of x, y coordinates) to a binary mask.
    """
    mask = np.zeros((height, width), dtype=np.uint8)  
    polygon = np.array(polygon, dtype=np.int32).reshape((-1, 2))  # Reshape to (N, 2)
    cv2.fillPoly(mask, [polygon], 1)  
    return mask

# Function to compute YOLO metrics
def compute_yolo_metrics(pred_boxes, pred_classes, gt_boxes, gt_classes, iou_threshold=0.5):
    """
    Compute YOLO metrics (precision, recall, F1-score) by matching predictions to ground truth.
    """
    tp = 0  # True positives
    fp = 0  # False positives
    fn = 0  # False negatives

    # Track which ground truth objects have been matched
    matched_gt = set()

    # For each predicted box, find the best matching ground truth box
    for i, pred_box in enumerate(pred_boxes):
        best_iou = 0
        best_match = -1
        for j, gt_box in enumerate(gt_boxes):
            if j in matched_gt:
                continue  # Skip already matched ground truth objects
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou and iou >= iou_threshold and pred_classes[i] == gt_classes[j]:
                best_iou = iou
                best_match = j
        if best_match != -1:
            tp += 1  # True positive
            matched_gt.add(best_match)
        else:
            fp += 1  # False positive

    # False negatives are ground truth objects that were not matched
    fn = len(gt_boxes) - len(matched_gt)

    # Compute precision, recall, and F1-score
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

    return precision, recall, f1

# Function to compute YOLO accuracy
def compute_yolo_accuracy(pred_boxes, pred_classes, gt_boxes, gt_classes, iou_threshold=0.5):
    """
    Compute YOLO accuracy based on IoU and class matching.
    """
    tp = 0  # True positives
    fp = 0  # False positives
    fn = 0  # False negatives

    # For each ground truth box, find the best matching predicted box
    matched_preds = set()  
    for i, gt_box in enumerate(gt_boxes):
        best_iou = 0
        best_match = -1
        for j, pred_box in enumerate(pred_boxes):
            if j in matched_preds:
                continue  # Skip already matched predictions
            iou = compute_iou(gt_box, pred_box)
            if iou > best_iou and iou >= iou_threshold and pred_classes[j] == gt_classes[i]:
                best_iou = iou
                best_match = j
        if best_match != -1:
            tp += 1  
            matched_preds.add(best_match)
        else:
            fn += 1  

    # False positives are predictions that were not matched to any ground truth
    fp = len(pred_boxes) - len(matched_preds)

    # Compute accuracy
    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) != 0 else 0
    return accuracy

# Function to compute mIoU and mAP-50 for SAM
def compute_miou(pred_mask, gt_mask):
    """
    Compute mean Intersection over Union (mIoU) for segmentation.
    """
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return intersection / union if union != 0 else 0

def compute_map50(pred_mask, gt_mask, iou_threshold=0.5):
    """
    Compute mean Average Precision at IoU threshold 0.50 (mAP-50) for segmentation.
    """
    # Flatten masks for pixel-wise comparison
    pred_mask_flat = pred_mask.flatten()
    gt_mask_flat = gt_mask.flatten()

    # Compute precision and recall
    precision = precision_score(gt_mask_flat, pred_mask_flat, zero_division=1)
    recall = recall_score(gt_mask_flat, pred_mask_flat, zero_division=1)

    # Compute AP (Average Precision)
    ap = precision if recall > 0 else 0
    return ap

# Iterate over all images in the folder
for image_name in os.listdir(images_folder):
    if not image_name.endswith((".jpg", ".png")):
        continue  # Skip non-image files

    image_path = os.path.join(images_folder, image_name)
    gt_label_path = os.path.join(labels_folder, image_name.replace(".jpg", ".txt").replace(".png", ".txt"))

    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Image not found: {image_path}")
        continue

    # Get image dimensions
    height, width = image.shape[:2]

    yolo_results = yolo_model(image_path, conf=0.25)  
    pred_boxes = yolo_results[0].boxes.xyxy.cpu().numpy()  
    pred_classes = yolo_results[0].boxes.cls.cpu().numpy()  

    # Load ground truth labels (polygon format)
    gt_boxes, gt_classes, gt_mask = [], [], np.zeros((height, width), dtype=np.uint8)
    if os.path.exists(gt_label_path):
        with open(gt_label_path, "r") as f:
            for line in f.readlines():
                parts = list(map(float, line.strip().split()))
                if len(parts) < 6:  # Skip invalid lines (polygons must have at least 3 points)
                    print(f"Skipping invalid line: {line}")
                    continue
                class_id = int(parts[0])  # Class ID
                polygon = parts[1:]  # Polygon coordinates (x1, y1, x2, y2, ...)
                # Convert polygon coordinates from normalized to absolute values
                polygon = [(int(x * width), int(y * height)) for x, y in zip(polygon[::2], polygon[1::2])]
                # Add to ground truth mask
                gt_mask = np.maximum(gt_mask, polygon_to_mask(polygon, width, height))
                # Convert polygon to bounding box
                x_coords = [x for x, y in polygon]
                y_coords = [y for x, y in polygon]
                x1 = min(x_coords)
                y1 = min(y_coords)
                x2 = max(x_coords)
                y2 = max(y_coords)
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(class_id)
    else:
        print(f"Label file not found: {gt_label_path}")
        continue

    gt_boxes = np.array(gt_boxes)
    gt_classes = np.array(gt_classes)

    if len(pred_classes) > 0 and len(gt_classes) > 0:
        precision, recall, f1 = compute_yolo_metrics(pred_boxes, pred_classes, gt_boxes, gt_classes)
        yolo_metrics = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": compute_yolo_accuracy(pred_boxes, pred_classes, gt_boxes, gt_classes),
        }
    else:
        print(f"No ground truth labels found for {image_name}. YOLO metrics will be 0.")
        yolo_metrics = {"precision": 0, "recall": 0, "f1": 0, "accuracy": 0}

    sam_results = sam_model(image, bboxes=pred_boxes, device="cpu")

    # Combine all predicted masks
    pred_mask = np.zeros_like(gt_mask, dtype=np.uint8)
    for mask in sam_results[0].masks.data:
        mask_np = mask.cpu().numpy().astype(np.uint8)
        pred_mask = np.maximum(pred_mask, mask_np)  # Merge masks

    # Compute Segmentation Metrics
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    sam_metrics = {
        "iou": intersection / union if union != 0 else 0,
        "dice": (2 * intersection) / (pred_mask.sum() + gt_mask.sum()) if (pred_mask.sum() + gt_mask.sum()) != 0 else 0,
        "accuracy": accuracy_score(gt_mask.flatten(), pred_mask.flatten()),
        "sensitivity": recall_score(gt_mask.flatten(), pred_mask.flatten(), zero_division=1),
        "specificity": recall_score(1 - gt_mask.flatten(), 1 - pred_mask.flatten(), zero_division=1),
        "miou": compute_miou(pred_mask, gt_mask),
        "map50": compute_map50(pred_mask, gt_mask),
    }

    for key in yolo_metrics:
        all_yolo_metrics[key].append(yolo_metrics[key])
    for key in sam_metrics:
        all_sam_metrics[key].append(sam_metrics[key])

avg_yolo_metrics = {key: np.mean(values) for key, values in all_yolo_metrics.items()}
avg_sam_metrics = {key: np.mean(values) for key, values in all_sam_metrics.items()}

print("\n Average YOLO Object Detection Metrics:")
print(f" Precision: {avg_yolo_metrics['precision']:.4f}")
print(f" Recall: {avg_yolo_metrics['recall']:.4f}")
print(f" F1-score: {avg_yolo_metrics['f1']:.4f}")
print(f" Accuracy: {avg_yolo_metrics['accuracy']:.4f}")

print("\n Average SAM Segmentation Metrics:")
print(f" IoU: {avg_sam_metrics['iou']:.4f}")
print(f" Dice Score: {avg_sam_metrics['dice']:.4f}")
print(f" Accuracy: {avg_sam_metrics['accuracy']:.4f}")
print(f" Sensitivity: {avg_sam_metrics['sensitivity']:.4f}")
print(f" Specificity: {avg_sam_metrics['specificity']:.4f}")
print(f" mIoU: {avg_sam_metrics['miou']:.4f}")
print(f" mAP-50: {avg_sam_metrics['map50']:.4f}")